## Esse algoritmo corrige e formata os dados, de maneira a agrupa-los nas cidades que compoem o cariri

In [7]:
import pandas as pd
from pathlib import Path

# Localiza onde o Scripts está e sobe um nível para a raiz do projeto (se o arquivo estiver renomeado, ele localiza onde estamos basicmamnte)

# Path.cwd() é a pasta 'Scripts', o .parent vai para 'analise_bnb'
RAIZ_PROJETO = Path.cwd().parent
# Define o caminho para a pasta de dados brutos
PASTA_BRUTOS = RAIZ_PROJETO / 'Dados_brutos'
# Define o caminho para a pasta de dados filtrados
PASTA_FILTRADOS = RAIZ_PROJETO / 'Dados_filtrados'

In [8]:
# Objetivo: recarregar o df com a acentuação correta, valores numericos sem serem considerados string e as datas corretas para as analises temporais

#Definindo o dataframe "quebrado" em questão
df_trimestre = pd.read_csv(PASTA_BRUTOS / 'Anual_2025' / 'Contratações FNE - Dezembro 2025.csv', sep =',' , encoding='utf-8')

# Remove espaços extras de todos os nomes de colunas
df_trimestre.columns = df_trimestre.columns.str.strip()

# Remove espaços extras nos nomes das cidades
df_trimestre['NM_MUN'] = df_trimestre['NM_MUN'].str.strip()

# Padroniza o nome da coluna (Se existir 'PORTE_S400', vira 'PORTE')
df_trimestre = df_trimestre.rename(columns={'PORTE_S400': 'PORTE'})

# Remove espaços extras na coluna: PORTE
df_trimestre['PORTE'] = df_trimestre['PORTE'].str.strip()

# Remove espaços extras na coluna: PROGRAMA
df_trimestre['PROGRAMA'] = df_trimestre['PROGRAMA'].str.strip()

# Remove espaços extras na coluna: ATIVIDADE(maior fator)
df_trimestre['ATIVIDADE(maior fator)'] = df_trimestre['ATIVIDADE(maior fator)'].str.strip()

def normaliza_numero_us(serie: pd.Series) -> pd.Series:
    return (
        serie
        .astype(str)
        .str.strip()
        .str.replace('"', '', regex=False)   # 1º Remove aspas acidentais, se houver
        .str.replace('.', '', regex=False)   # 2º Remove todos os pontos de milhar
        .str.replace(',', '.', regex=False)  # 3º Troca a vírgula dos centavos por ponto
        .astype(float)                       # 4º Converte com segurança para decimal
    )

df_trimestre['VALOR_CTR'] = normaliza_numero_us(df_trimestre['VALOR_CTR'])
df_trimestre['QTD']       = normaliza_numero_us(df_trimestre['QTD'])


In [ ]:
#Criação de uma variavel auxliar para conter cidades de interesse para a analise (no nosso caso é a região do Cariri)
cidades_alvo = [
    'JUAZEIRO DO NORTE', 'CRATO', 'BARBALHA',
    'ABAIARA', 'ALTANEIRA', 'ANTONINA DO NORTE', 'ARARIPE', 'ASSARÉ', 'ASSARE',
    'AURORA', 'BARRO', 'BREJO SANTO', 'CAMPOS SALES', 'CARIRIAÇU', 'CARIRIACU',
    'FARIAS BRITO', 'GRANJEIRO', 'JARDIM', 'JATI', 'LAVRAS DA MANGABEIRA', 
    'MAURITI', 'MILAGRES', 'MISSÃO VELHA', 'NOVA OLINDA', 'PENAFORTE', 
    'PORTEIRAS', 'POTENGI', 'SALITRE', 'SANTANA DO CARIRI', 'TARRAFAS', 
    'VÁRZEA ALEGRE', 'VARZEA ALEGRE'
]
    
programas_alvo = [
    'PRONAF MULHER GRUPO B SEMIARIDO', 
    'PRONAF MULHER GRUPO B SEMIARID',
    'PRONAF-B/PLANO-SAFRA SEMIARIDO',
    'PRONAF-MAIS ALIMENTOS (FNE)',
    'PRONAF GRUPO ""B"" - FNE',
    'PRONAF SEMI-ARIDO - FNE',
    'PRONAF MULHER (GRUPO A)',
    'PRONAF MULHER (GRUPO B)',       
    'PRONAF GRUPO "B" - FNE',   
    'PRONAF GRUPO "A" - FNE',
    'PRONAF FLORESTA - FNE',
    'PRONAF MULHER - FNE',
    'PRONAF-COMUM (FNE)',
    'PRONAF-ECO (FNE)'  
    
]

#Criando outro dataframe utilizando o dataframe original porem com o filtro da cidade que queremos analisar
# Criando o dataframe filtrando pela cidade alvo E pelo estado do Ceará
df_anual_2025_cidades = df_trimestre[ (df_trimestre['NM_MUN'].isin(cidades_alvo)) & (df_trimestre['UF_MUN'] == 'CE') & (df_trimestre['PROGRAMA'].isin(programas_alvo)) ]


#Salvando o novo dataframe filtrado na pasta dos arquivos filtrados, para ser utilizado nas analises
df_anual_2025_cidades.to_csv(PASTA_FILTRADOS / '2025' / 'anual_2025_cariri.csv', index=False, encoding='utf-8')
